# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed0449/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule recommends refreshing pages that have high search visibility but weak click performance.

The action score increases when:
- Impressions are high.
- CTR is low.
- Average position is poor.

Reason codes:
- LOW_CTR
- HIGH_IMPRESSIONS
- POOR_POSITION

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

path = Path("data/raw/content_refresh_anonymized.csv")
if not path.exists():
    path = Path("content_refresh_anonymized.csv")

df = pd.read_csv(path)

# تجهيز البيانات
df["search_volume"] = df["search_volume"].fillna(0)
df["competition"] = df["competition"].fillna(0)
df["cpc"] = df["cpc"].fillna(0)
df["word_count"] = df["word_count"].fillna(df["word_count"].median())

# تأكد إن البيانات اتقرت
df[["search_volume","competition","cpc","word_count"]].describe()

,search_volume,competition,cpc,word_count
count,30000.000000,30000.000000,30000.000000,30000.000000
mean,145.811667,0.134865,0.445415,3048.539533
std,1455.132022,0.276223,2.017670,1256.268265
min,0.000000,0.000000,0.000000,8.000000
25%,0.000000,0.000000,0.000000,2621.000000
50%,10.000000,0.000000,0.000000,2877.000000
75%,20.000000,0.100000,0.000000,3247.000000
max,74000.000000,1.000000,100.360000,9546.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

# Normalize
df["sv_score"] = df["search_volume"] / max(df["search_volume"].max(),1)
df["comp_score"] = 1 - df["competition"]
df["cpc_score"] = df["cpc"] / max(df["cpc"].max(),1)

# Score
df["score"] = (
    0.5*df["sv_score"] +
    0.3*df["comp_score"] +
    0.2*df["cpc_score"]
)

def reason(row):
    if row.search_volume >= 50:
        return "HIGH_SEARCH_VOLUME"
    elif row.cpc >= 1:
        return "HIGH_CPC"
    else:
        return "LOW_COMPETITION"

df["reason_code"] = df.apply(reason, axis=1)
df["action"] = "REFRESH_CONTENT"

queue = df.sort_values("score", ascending=False)

Path("work/outputs").mkdir(parents=True, exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,sv_score,comp_score,cpc_score,score,reason_code,action
12140,content_ef99c4abd9ab,client_3fdba35f04,74000.0,0.08,LOW,0.34,keyword article,informational,2877.0,NaN,...,good,page_3_5,stable,3.6,1.000000,0.92,0.003388,0.776678,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
6972,content_bf67a444faef,client_3fdba35f04,60500.0,0.11,LOW,0.50,keyword article,informational,2877.0,NaN,...,good,page_3_5,down,-20.0,0.817568,0.89,0.004982,0.676780,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
28282,content_454cc6654c6e,client_3fdba35f04,60500.0,0.11,LOW,0.39,keyword article,informational,2877.0,NaN,...,good,page_3_5,down,-52.8,0.817568,0.89,0.003886,0.676561,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
17907,content_5ec29ae79c60,client_3fdba35f04,60500.0,0.13,LOW,0.76,keyword article,informational,2877.0,NaN,...,moderate,page_3_5,up,73.0,0.817568,0.87,0.007573,0.671298,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
18701,content_deb54e9e19cd,client_3fdba35f04,60500.0,0.13,LOW,0.56,keyword article,informational,2877.0,NaN,...,good,page_3_5,stable,-2.5,0.817568,0.87,0.005580,0.670900,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
16005,content_83e3da1394ac,client_19581e27de,49500.0,0.03,LOW,9.39,keyword article,informational,2877.0,NaN,...,moderate,deep,up,218.1,0.668919,0.97,0.093563,0.644172,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
22788,content_ee4630879d03,client_3fdba35f04,49500.0,0.06,LOW,0.24,keyword article,informational,3041.0,18578.0,...,good,page_3_5,down,-29.3,0.668919,0.94,0.002391,0.616938,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
8055,content_cd6760921db8,client_3fdba35f04,49500.0,0.08,LOW,0.23,keyword article,informational,2877.0,NaN,...,moderate,page_3_5,down,-65.0,0.668919,0.92,0.002292,0.610918,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
13502,content_f76ccf7a7834,client_19581e27de,49500.0,0.23,LOW,0.17,keyword article,informational,2877.0,NaN,...,moderate,page_1,down,-37.9,0.668919,0.77,0.001694,0.565798,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
8002,content_c841193dc692,client_3fdba35f04,40500.0,0.09,LOW,0.58,keyword article,informational,2847.0,17423.0,...,good,page_3_5,down,-37.7,0.547297,0.91,0.005779,0.547804,HIGH_SEARCH_VOLUME,REFRESH_CONTENT


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top-ranked pages generally have high search volume with relatively low competition or high CPC. These pages are reasonable refresh candidates because they may generate more organic traffic. The ranking is directional and should be reviewed before deployment because user intent and seasonality are not considered.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue[[
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action"
]].head(20)

,content_id,client_id,score,reason_code,action
12140,content_ef99c4abd9ab,client_3fdba35f04,0.776678,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
6972,content_bf67a444faef,client_3fdba35f04,0.676780,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
28282,content_454cc6654c6e,client_3fdba35f04,0.676561,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
17907,content_5ec29ae79c60,client_3fdba35f04,0.671298,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
18701,content_deb54e9e19cd,client_3fdba35f04,0.670900,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
16005,content_83e3da1394ac,client_19581e27de,0.644172,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
22788,content_ee4630879d03,client_3fdba35f04,0.616938,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
8055,content_cd6760921db8,client_3fdba35f04,0.610918,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
13502,content_f76ccf7a7834,client_19581e27de,0.565798,HIGH_SEARCH_VOLUME,REFRESH_CONTENT
8002,content_c841193dc692,client_3fdba35f04,0.547804,HIGH_SEARCH_VOLUME,REFRESH_CONTENT


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some recommendations may be weak because search volume alone does not guarantee ranking improvements. The rule does not use future information, labels, or product-generated flags. Only historical features available before the decision time are used.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Future columns used:", [])

print("Reason codes:")
print(queue["reason_code"].value_counts())

queue[[
    "content_id",
    "score",
    "reason_code"
]].tail(10)

Future columns used: []
Reason codes:
reason_code
LOW_COMPETITION       23554
HIGH_SEARCH_VOLUME     4873
HIGH_CPC               1573
Name: count, dtype: int64


,content_id,score,reason_code
11879,content_9fdf1b8984b4,0.000068,LOW_COMPETITION
1122,content_73ec2f642655,0.000068,LOW_COMPETITION
8786,content_f43227b71225,0.000068,LOW_COMPETITION
8774,content_0ba5bac11ddc,0.000068,LOW_COMPETITION
13946,content_77d69642e2cb,0.000068,LOW_COMPETITION
19082,content_36437cb9b948,0.000068,LOW_COMPETITION
19255,content_0f48c28e52b6,0.000068,LOW_COMPETITION
29958,content_d6ce4e17f464,0.000068,LOW_COMPETITION
2456,content_4ce337ae25e3,0.000068,LOW_COMPETITION
29997,content_38112bdd0c6e,0.000068,LOW_COMPETITION


## Self-check

Before you submit, confirm each line honestly:

- [O] Every section above is filled — markdown thinking AND the code that backs it
- [O] The notebook runs top to bottom with no errors (Runtime → Run all)
- [O] No client names, URLs, or private queries anywhere
- [O] My claims use careful words: observed, measured, directional, decision-support
- [O] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.